# TP : Optimisation d'un Réseau de Neurones FROM SCRATCH
## Mission : Améliorer la précision avec vos propres fonctions

---

** Contexte :**
Dans le cours, vous avez implémenté from scratch toutes les fonctions d'un réseau de neurones :
- Fonctions d'activation (sigmoid, relu)
- Initialisation des paramètres
- Forward propagation
- Calcul du coût
- Backward propagation
- Mise à jour des paramètres
- Entraînement complet

** Objectif :**
Vous allez maintenant **réutiliser ces fonctions** sur un nouveau dataset (classification binaire de cancer du sein) et **optimiser le réseau** pour obtenir la meilleure accuracy possible.

** Challenge :**
Atteindre **95% d'accuracy** sur le test set !

**⏱️ Durée :** 1h30 - 2h

---

## PARTIE 0 : Rappel des Fonctions (Copier depuis le cours)

**TODO :** Copiez ici toutes les fonctions que vous avez implémentées pendant le cours

**Fonctions nécessaires :**
1. `sigmoid(z)` et `sigmoid_derivative(z)`
2. `relu(z)` et `relu_derivative(z)`
3. `initialize_parameters(layer_dims)`
4. `forward_propagation(X, parameters, activation_hidden)`
5. `compute_cost(AL, Y)`
6. `backward_propagation(AL, Y, parameters, cache, activation_hidden)`
7. `update_parameters(parameters, gradients, learning_rate)`
8. `train_neural_network(...)`
9. `predict(X, parameters, activation_hidden)`
10. `compute_accuracy(predictions, Y)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

# ========== 1. FONCTIONS D'ACTIVATION ==========
def sigmoid(z):
    """Sortie entre 0 et 1 (proba)"""
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    """Max(0, z) - Standard pour couches cachées"""
    return np.maximum(0, z)

def relu_derivative(z):
    """1 si z > 0 sinon 0"""
    return (z > 0).astype(float)

# ========== 2. INITIALISATION ==========
def initialize_parameters(layer_dims):
    """Initialisation de He pour ReLU"""
    np.random.seed(42)
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        # Initialisation de He (recommandé pour ReLU)
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2 / layer_dims[l-1]) 
        # Biais à zéro
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

# ========== 3. FORWARD PROPAGATION ==========
def forward_propagation(X, parameters, activation_hidden='relu'):
    cache = {}
    A = X.T # Transpose pour avoir (n_features, m_samples)
    L = len(parameters) // 2

    # Couches cachées
    for l in range(1, L):
        A_prev = A 
        W = parameters['W' + str(l)]
        b = parameters['b' + str(l)]
        
        Z = np.dot(W, A_prev) + b
        
        if activation_hidden == 'relu':
            A = relu(Z)
        else: # sigmoid
            A = sigmoid(Z)
            
        cache['A' + str(l-1)] = A_prev
        cache['Z' + str(l)] = Z

    # Couche de sortie (Toujours Sigmoid pour classification binaire)
    W = parameters['W' + str(L)]
    b = parameters['b' + str(L)]
    ZL = np.dot(W, A) + b
    AL = sigmoid(ZL)

    cache['A' + str(L-1)] = A
    cache['Z' + str(L)] = ZL
    cache['AL'] = AL

    return AL, cache

# ========== 4. COÛT ==========
def compute_cost(AL, Y):
    m = Y.shape[0]
    Y = Y.T # Pour matcher AL

    # Ajout d'epsilon pour éviter log(0)
    epsilon = 1e-8
    cost = -1/m * np.sum(Y * np.log(AL + epsilon) + (1-Y) * np.log(1-AL + epsilon))
    return np.squeeze(cost)

# ========== 5. BACKWARD PROPAGATION ==========
def backward_propagation(AL, Y, parameters, cache, activation_hidden='relu'):
    gradients = {}
    L = len(parameters) // 2
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) 

    # Gradient sortie
    dZL = AL - Y # Dérivée simplifiée BCE + Sigmoid
    gradients["dW" + str(L)] = 1./m * np.dot(dZL, cache['A' + str(L-1)].T)
    gradients["db" + str(L)] = 1./m * np.sum(dZL, axis=1, keepdims=True)
    
    dA_prev = np.dot(parameters['W' + str(L)].T, dZL)

    # Gradients couches cachées
    for l in reversed(range(1, L)):
        Z = cache['Z' + str(l)]

        if activation_hidden == 'relu':
            dZ = dA_prev * relu_derivative(Z)
        else:
            dZ = dA_prev * sigmoid_derivative(Z)
            
        gradients["dW" + str(l)] = 1./m * np.dot(dZ, cache['A' + str(l-1)].T)
        gradients["db" + str(l)] = 1./m * np.sum(dZ, axis=1, keepdims=True)

        if l > 1:
            dA_prev = np.dot(parameters['W' + str(l)].T, dZ)

    return gradients

# ========== 6. UPDATE ==========
def update_parameters(parameters, gradients, learning_rate):
    L = len(parameters) // 2 
    for l in range(1, L + 1):
        parameters["W" + str(l)] = parameters["W" + str(l)] - learning_rate * gradients["dW" + str(l)]
        parameters["b" + str(l)] = parameters["b" + str(l)] - learning_rate * gradients["db" + str(l)]
    return parameters

# ========== 7. ENTRAÎNEMENT ==========
def train_neural_network(X, Y, layer_dims, learning_rate=0.01, num_iterations=1000, activation_hidden='relu', print_cost=True):
    parameters = initialize_parameters(layer_dims)
    costs = []

    for i in range(num_iterations):
        AL, cache = forward_propagation(X, parameters, activation_hidden)
        cost = compute_cost(AL, Y)
        gradients = backward_propagation(AL, Y, parameters, cache, activation_hidden)
        parameters = update_parameters(parameters, gradients, learning_rate)

        if i % 100 == 0:
            costs.append(cost)
            if print_cost:
                print(f"Cost after iteration {i}: {cost:.6f}")

    return parameters, costs

# ========== 8. PRÉDICTION ==========
def predict(X, parameters, activation_hidden='relu'):
    AL, _ = forward_propagation(X, parameters, activation_hidden)
    predictions = (AL > 0.5).astype(int)
    return predictions.T 

def compute_accuracy(predictions, Y):
    return (100 - np.mean(np.abs(predictions - Y)) * 100)

print(" Fonctions importées !")


---

## PARTIE 1 : Chargement du Dataset

### Dataset : Breast Cancer Wisconsin

**Description :**
- Classification binaire : tumeur **bénigne (0)** ou **maligne (1)**
- 569 exemples
- 30 features (caractéristiques des cellules)
- Features : rayon, texture, périmètre, aire, etc.

**Enjeu médical :** Détecter les cancers malins avec la meilleure précision possible !

### Exercice 1.1 : Charger et explorer les données

**TODO :**
1. Charger le dataset avec `load_breast_cancer()`
2. Afficher les informations principales
3. Analyser la distribution des classes

In [ ]:
# TODO: Charger le dataset
data = load_breast_cancer()
X = data.data
y = data.target

print(" INFORMATIONS DU DATASET")
print("="*60)
# TODO: Afficher
print(f"Nombre d'exemples : {X.shape[0]}")
print(f"Nombre de features : {X.shape[1]}")
print(f"Nombre de classes : {len(np.unique(y))}")
print(f"\nClasses : ")
print(f"  0 (Maligne) : {np.sum(y==0)} exemples ({np.sum(y==0)/len(y)*100:.1f}%)")
print(f"  1 (Bénigne)  : {np.sum(y==1)} exemples ({np.sum(y==1)/len(y)*100:.1f}%)")
# Note: In sklearn breast cancer dataset: 0 = Malignant, 1 = Benign

# TODO: Afficher les noms des features
print(f"\n📋 Exemples de features :")
print(data.feature_names[:10])

In [ ]:
# TODO: Visualiser la distribution des classes
plt.figure(figsize=(8, 6))
# Votre code ici (bar plot ou pie chart)
count_malignant = np.sum(y==0)
count_benign = np.sum(y==1)

plt.bar(['Maligne (0)', 'Bénigne (1)'], [count_malignant, count_benign], color=['red', 'green'])
plt.ylabel('Nombre d\'exemples')

plt.title('Distribution des Classes', fontsize=14, fontweight='bold')
plt.show()

**❓ Question 1.1 :** Le dataset est-il équilibré ? Pourquoi est-ce important ?

*Votre réponse :*




### Exercice 1.2 : Préparation des données

**TODO :**
1. Split train/test (80/20)
2. Normaliser avec StandardScaler
3. Vérifier les dimensions

In [ ]:
# TODO: Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: Normalisation (CRUCIAL)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # Fit and transform on train
X_test = scaler.transform(X_test)       # Only transform on test

# Reshape y pour avoir (m, 1)
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print("✅ DONNÉES PRÉPARÉES")
print(f"   X_train : {X_train.shape}")
print(f"   y_train : {y_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_test  : {y_test.shape}")

---

## PARTIE 2 : Modèle Baseline (Point de Départ)

### Exercice 2.1 : Premier modèle simple

**TODO :**
Entraîner un modèle baseline avec :
- Architecture : `[30, 10, 1]` (30 inputs, 1 couche cachée de 10 neurones, 1 output)
- Activation : `relu`
- Learning rate : `0.01`
- Iterations : `1000`

In [ ]:
print(" ENTRAÎNEMENT MODÈLE BASELINE\n")

# TODO: Entraîner le modèle baseline
layer_dims_baseline = [30, 10, 1]

# Entraînement
parameters_baseline, costs_baseline = train_neural_network(X_train, y_train, layer_dims_baseline, learning_rate=0.01, num_iterations=1000)

# TODO: Prédictions
y_pred_train_baseline = predict(X_train, parameters_baseline)
y_pred_test_baseline = predict(X_test, parameters_baseline)

# TODO: Accuracy
baseline_train_acc = compute_accuracy(y_pred_train_baseline, y_train)
baseline_test_acc = compute_accuracy(y_pred_test_baseline, y_test)

print("\n" + "="*60)
print("📊 RÉSULTATS BASELINE")
print("="*60)
print(f"Architecture : {layer_dims_baseline}")
print(f"Activation : relu")
print(f"Learning Rate : 0.01")
print(f"Iterations : 1000")
print(f"\nAccuracy Train : {baseline_train_acc:.2f}%")
print(f"Accuracy Test  : {baseline_test_acc:.2f}%")
print(f"Gap (overfitting) : {baseline_train_acc - baseline_test_acc:.2f}%")

In [ ]:
# TODO: Visualiser la courbe d'apprentissage
plt.figure(figsize=(10, 6))
plt.plot(costs_baseline, 'b-', linewidth=2)
plt.xlabel('Itérations (x100)', fontsize=12)
plt.ylabel('Coût', fontsize=12)
plt.title('Courbe d\'Apprentissage - Baseline', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

### Exercice 2.2 : Analyse du modèle baseline

**TODO :**
1. Matrice de confusion
2. Rapport de classification

In [ ]:
# TODO: Matrice de confusion
cm_baseline = confusion_matrix(y_test, y_pred_test_baseline)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Maligne', 'Bénigne'], # Attention ordre inversé scikit-learn
            yticklabels=['Maligne', 'Bénigne'])
plt.title('Matrice de Confusion - Baseline', fontsize=14, fontweight='bold')
plt.ylabel('Vraie Classe')
plt.xlabel('Classe Prédite')
plt.show()

# TODO: Rapport de classification
print("\ RAPPORT DE CLASSIFICATION")
print(classification_report(y_test, y_pred_test_baseline, target_names=['Maligne', 'Bénigne']))

**❓ Question 2.1 :** 
- Quels sont les **faux positifs** (bénignes prédites malignes) ?
- Quels sont les **faux négatifs** (malignes prédites bénignes) ?
- Lequel est plus grave d'un point de vue médical ?

*Votre réponse :*




---

## PARTIE 3 : Optimisation de l'Architecture

### Exercice 3.1 : Tester différentes architectures

**Objectif :** Trouver la meilleure architecture pour ce problème

**TODO :**
Tester au moins 5 architectures différentes.

**Suggestions :**
- `[30, 20, 1]` - Plus de neurones dans la couche cachée
- `[30, 15, 10, 1]` - 2 couches cachées
- `[30, 30, 15, 1]` - Réduction progressive
- `[30, 50, 25, 1]` - Architecture large
- Une architecture de votre choix !

**Garder constants :**
- activation='relu'
- learning_rate=0.01
- num_iterations=1000

In [ ]:
# TODO: Définir les architectures à tester
architectures = [
    [30, 10, 1],       # Baseline
    [30, 20, 1],       # Plus large
    [30, 15, 10, 1],   # 2 couches cachées
    [30, 30, 15, 1],   # Pyramide
    [30, 50, 25, 1],   # Large et profond
    [30, 10, 5, 1]     # Petit et profond
]

# Stockage des résultats
results_arch = {}

print(" TEST DES ARCHITECTURES\n")
print("="*70)

# TODO: Pour chaque architecture
for arch in architectures:
    arch_str = str(arch)
    print(f"\nTest : {arch_str}")
    
    # Entraîner
    params, costs = train_neural_network(X_train, y_train, arch, learning_rate=0.01, num_iterations=1000, print_cost=False)
    
    # Évaluer
    y_pred_test = predict(X_test, params)
    test_acc = compute_accuracy(y_pred_test, y_test)
    
    # Stocker
    results_arch[arch_str] = {
        'accuracy': test_acc,
        'costs': costs,
        'parameters': params
    }
    
    print(f"   Accuracy Test : {test_acc:.2f}%")

# Afficher le résumé
print("\n" + "="*70)
print(" RÉSUMÉ DES ARCHITECTURES")
print("="*70)
for arch_name, res in results_arch.items():
    print(f"{arch_name:<30} : {res['accuracy']:.2f}%")

In [ ]:
# TODO: Visualiser les performances
plt.figure(figsize=(12, 6))

# Graphique 1: Accuracies
architectures_names = list(results_arch.keys())
accuracies = [results_arch[arch]['accuracy'] for arch in architectures_names]

# Bar plot
bars = plt.bar(architectures_names, accuracies, color='skyblue')

# Ajouter les valeurs sur les barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom')

plt.xlabel('Architecture')
plt.ylabel('Accuracy Test (%)')
plt.title('Comparaison des Architectures', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylim(80, 100) # Zoom sur les bonnes perfs
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

**❓ Question 3.1 :** Quelle architecture donne les meilleurs résultats ? Pourquoi selon vous ?

*Votre réponse :*




---

## PARTIE 4 : Optimisation du Learning Rate

### Exercice 4.1 : Impact du Learning Rate

**TODO :**
Avec la meilleure architecture trouvée, tester différents learning rates :
- `[0.001, 0.005, 0.01, 0.05, 0.1, 0.5]`

In [ ]:
# TODO: Choisir la meilleure architecture
best_arch = [30, 20, 10, 1] # Meilleure architecture trouvée (20+10 neurones)

learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
results_lr = {}

print(f" TEST DES LEARNING RATES (Architecture: {best_arch})\n")
print("="*70)

# TODO: Tester chaque learning rate
for lr in learning_rates:
    print(f"\nTest LR : {lr}")
    
    # Entraîner
    params, costs = train_neural_network(X_train, y_train, best_arch, learning_rate=lr, num_iterations=1000, print_cost=False)
    
    # Évaluer
    y_pred_test = predict(X_test, params)
    test_acc = compute_accuracy(y_pred_test, y_test)
    
    # Stocker
    results_lr[lr] = {
        'accuracy': test_acc,
        'costs': costs
    }
    
    print(f"   Accuracy Test : {test_acc:.2f}%")

# Meilleur LR
best_lr = max(results_lr, key=lambda x: results_lr[x]['accuracy'])
print(f"\n🏆 Meilleur Learning Rate : {best_lr} ({results_lr[best_lr]['accuracy']:.2f}%)")

In [ ]:
# TODO: Visualiser les courbes de coût
plt.figure(figsize=(12, 6))

# Tracer la courbe de coût pour chaque LR
for lr, data in results_lr.items():
    plt.plot(data['costs'], label=f'LR={lr}')

plt.xlabel('Itérations (x100)')
plt.ylabel('Coût')
plt.title('Courbes de Coût - Impact du Learning Rate', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1) # Zoom pour mieux voir
plt.show()

**❓ Question 4.1 :** 
- Que se passe-t-il avec un LR trop grand (ex: 0.5) ?
- Que se passe-t-il avec un LR trop petit (ex: 0.001) ?
- Quel est le meilleur compromis ?

*Votre réponse :*




---

## PARTIE 5 : Optimisation du Nombre d'Itérations

### Exercice 5.1 : Trouver le bon nombre d'itérations

**TODO :**
Avec la meilleure architecture et le meilleur LR, tester :
- `[500, 1000, 2000, 3000, 5000]` itérations

Objectif : trouver le point où le modèle a convergé (plus d'itérations = plus de temps mais pas forcément meilleur)

In [ ]:
iterations_list = [500, 1000, 2000, 3000, 5000]
results_iter = {}

print(f" TEST DU NOMBRE D'ITÉRATIONS\n")
print(f"Architecture: {best_arch}")
print(f"Learning Rate: {best_lr}\n")
print("="*70)

# TODO: Tester chaque nombre d'itérations
for num_iter in iterations_list:
    print(f"\nTest {num_iter} itérations")
    
    # Entraîner
    params, costs = train_neural_network(X_train, y_train, best_arch, learning_rate=best_lr, num_iterations=num_iter, print_cost=False)
    
    # Évaluer
    y_pred_train = predict(X_train, params)
    y_pred_test = predict(X_test, params)
    train_acc = compute_accuracy(y_pred_train, y_train)
    test_acc = compute_accuracy(y_pred_test, y_test)
    
    # Stocker
    results_iter[num_iter] = {
        'train_acc': train_acc,
        'test_acc': test_acc,
        'gap': train_acc - test_acc,
        'costs': costs
    }
    
    print(f"   Train: {train_acc:.2f}% | Test: {test_acc:.2f}% | Gap: {train_acc-test_acc:.2f}%")

# Meilleur nombre d'itérations (max test acc)
best_iter = max(results_iter, key=lambda x: results_iter[x]['test_acc'])
print(f"\n🏆 Meilleur nombre d'itérations : {best_iter} ({results_iter[best_iter]['test_acc']:.2f}%)")

In [ ]:
# TODO: Visualiser l'évolution des performances
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

iters = list(results_iter.keys())
test_accs = [results_iter[i]['test_acc'] for i in iters]
gaps = [results_iter[i]['gap'] for i in iters]

# Graphique 1: Accuracy vs Iterations
axes[0].plot(iters, test_accs, 'o-', color='green', linewidth=2)
axes[0].set_xlabel('Itérations')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Accuracy Test vs Itérations')
axes[0].grid(True, alpha=0.3)

# Graphique 2: Gap (overfitting) vs Iterations
axes[1].plot(iters, gaps, 'o-', color='red', linewidth=2)
axes[1].set_xlabel('Itérations')
axes[1].set_ylabel('Gap Train-Test (%)')
axes[1].set_title('Overfitting (Gap) vs Itérations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**❓ Question 5.1 :** À partir de combien d'itérations observez-vous de l'overfitting ? Comment le détectez-vous ?

*Votre réponse :*




---

## PARTIE 6 : Comparaison des Fonctions d'Activation

### Exercice 6.1 : ReLU vs Sigmoid dans les couches cachées

**TODO :**
Comparer les performances avec :
- `activation_hidden='relu'`
- `activation_hidden='sigmoid'`

Utiliser : meilleure architecture, meilleur LR, meilleur nombre d'itérations

In [ ]:
activations = ['relu', 'sigmoid']
results_activation = {}

print(" COMPARAISON DES ACTIVATIONS\n")
print(f"Architecture: {best_arch}")
print(f"Learning Rate: {best_lr}")
print(f"Iterations: {best_iter}\n")
print("="*70)

# TODO: Tester chaque activation
for activation in activations:
    print(f"\nTest activation : {activation}")
    
    # Entraîner
    params, costs = train_neural_network(X_train, y_train, best_arch, learning_rate=best_lr, num_iterations=best_iter, activation_hidden=activation, print_cost=False)
    
    # Évaluer
    y_pred_test = predict(X_test, params, activation_hidden=activation)
    test_acc = compute_accuracy(y_pred_test, y_test)
    
    # Stocker
    results_activation[activation] = {
        'accuracy': test_acc,
        'costs': costs,
        'parameters': params
    }
    
    print(f"   Accuracy Test : {test_acc:.2f}%")

# Meilleure activation
best_activation = max(results_activation, key=lambda x: results_activation[x]['accuracy'])
print(f"\n🏆 Meilleure activation : {best_activation} ({results_activation[best_activation]['accuracy']:.2f}%)")

In [ ]:
# TODO: Comparer les courbes de coût
plt.figure(figsize=(12, 6))

for act, res in results_activation.items():
    plt.plot(res['costs'], label=f'Activation: {act}')

plt.xlabel('Itérations (x100)')
plt.ylabel('Coût')
plt.title('Comparaison ReLU vs Sigmoid', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**❓ Question 6.1 :** Quelle activation converge le plus vite ? Laquelle donne la meilleure accuracy ? Pourquoi ?

*Votre réponse :*




---

## PARTIE 7 : Modèle Final Optimisé

### Exercice 7.1 : Créer le modèle final

**TODO :**
Combiner TOUS les meilleurs hyperparamètres trouvés

In [ ]:
print(" CRÉATION DU MODÈLE FINAL OPTIMISÉ\n")
print("="*70)

# TODO: Récapitulatif
print(" HYPERPARAMÈTRES OPTIMAUX CHECK:")
print(f"   Architecture : {best_arch}")
print(f"   Activation : {best_activation}")
print(f"   Learning Rate : {best_lr}")
print(f"   Iterations : {best_iter}")

# Entraînement Final
final_params, final_costs = train_neural_network(
    X_train, y_train, 
    best_arch, 
    learning_rate=best_lr, 
    num_iterations=best_iter, 
    activation_hidden=best_activation,
    print_cost=True
)

# Évaluation Finale
final_train_pred = predict(X_train, final_params, best_activation)
final_test_pred = predict(X_test, final_params, best_activation)
final_train_acc = compute_accuracy(final_train_pred, y_train)
final_test_acc = compute_accuracy(final_test_pred, y_test)

print("\n" + "="*70)
print(f" RÉSULTAT FINAL: {final_test_acc:.2f}%")
print("="*70)
if final_test_acc >= 95:
    print("✅ OBJECTIF > 95% ATTEINT ! BRAVO !")
else:
    print("⚠️ Objectif non atteint. Essayez d'augmenter les itérations ou changer l'architecture.")

### Exercice 7.2 : Évaluation médicale du modèle final

**TODO :**
Analyser les performances du point de vue médical

In [ ]:
# TODO: Matrice de confusion
cm_baseline = confusion_matrix(y_test, y_pred_test_baseline)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Maligne', 'Bénigne'], # Attention ordre inversé scikit-learn
            yticklabels=['Maligne', 'Bénigne'])
plt.title('Matrice de Confusion - Baseline', fontsize=14, fontweight='bold')
plt.ylabel('Vraie Classe')
plt.xlabel('Classe Prédite')
plt.show()

# TODO: Rapport de classification
print("\ RAPPORT DE CLASSIFICATION")
print(classification_report(y_test, y_pred_test_baseline, target_names=['Maligne', 'Bénigne']))

**❓ Question 7.1 :** 
- Votre modèle minimise-t-il les faux négatifs (le plus dangereux) ?
- Que pourrait-on faire pour réduire encore plus les faux négatifs ?

*Votre réponse :*




### Exercice 7.3 : Comparaison Baseline vs Final

**TODO :**
Créer un graphique comparatif

In [ ]:
# TODO: Comparaison visuelle
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Graphique 1: Courbes de coût
axes[0].plot(costs_baseline, 'b-', label='Baseline', linewidth=2)
axes[0].plot(costs_final, 'g-', label='Final Optimisé', linewidth=2)
axes[0].set_xlabel('Itérations (x100)')
axes[0].set_ylabel('Coût')
axes[0].set_title('Comparaison des Courbes de Coût', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Graphique 2: Accuracies
models = ['Baseline', 'Final']
accuracies = [baseline_test_acc, final_test_acc]
colors = ['#3498db', '#2ecc71']

bars = axes[1].bar(models, accuracies, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Accuracy Test (%)')
axes[1].set_title('Amélioration de la Performance', fontweight='bold')
axes[1].set_ylim([80, 100])
axes[1].grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs sur les barres
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---

## PARTIE 8 : Synthèse et Réflexion

### Récapitulatif de votre travail

**TODO :** Remplissez ce tableau

| Hyperparamètre | Baseline | Final Optimisé |
|----------------|----------|----------------|
| Architecture | [30, 10, 1] | _____________ |
| Activation | relu | _____________ |
| Learning Rate | 0.01 | _____________ |
| Iterations | 1000 | _____________ |
| **Test Accuracy** | _____% | _____% |
| **Amélioration** | - | **+_____%** |

### Questions de Réflexion Finales

**❓ Question 8.1 :** Résumez les 3 optimisations les plus impactantes que vous avez faites

*Votre réponse :*

1. 

2. 

3. 


**❓ Question 8.2 :** Quelle est la différence entre implémenter from scratch vs utiliser Scikit-Learn ?

*Avantages from scratch :*


*Inconvénients from scratch :*



**❓ Question 8.3 :** Dans un contexte médical réel, que faudrait-il améliorer avant de déployer ce modèle ?

*Votre réponse :*



**❓ Question 8.4 :** Si vous aviez accès à plus de données (10x plus), comment changeriez-vous votre approche ?

*Votre réponse :*




---

## BONUS : Défis Supplémentaires (Optionnel)

### Défi 1 : Atteindre 97% d'accuracy

**Pistes :**
- Augmenter encore plus le nombre d'itérations
- Tester des architectures plus profondes
- Implémenter une diminution progressive du learning rate (learning rate decay)
- Implémenter le momentum

In [ ]:
# Votre code pour le défi 1



### Défi 2 : Visualiser les poids du réseau

**TODO :** Visualiser quelles features sont les plus importantes en analysant les poids de la première couche

In [ ]:
# Votre code pour le défi 2
# Indice: parameters_final['W1'] contient les poids de la première couche



### Défi 3 : Implémenter un Learning Rate Decay

**TODO :** Modifier la fonction `train_neural_network` pour diminuer progressivement le learning rate

**Formule :** $\alpha_t = \frac{\alpha_0}{1 + \text{decay_rate} \times t}$

In [ ]:
# Votre code pour le défi 3

